# Qutrit GST

Two ions in one trap, read out by collecting fluorescence from both at once, are not a two-qubit system. The detector counts photons; it cannot say which ion emitted them. Three outcomes come back: no ion bright, one bright, both bright. The states that produce them span the symmetric subspace of the two-qubit Hilbert space, which is three-dimensional, so the thing to characterize is a qutrit, and the third level is a third of the system rather than somewhere population escapes to.

In [1]:
import numpy as np
from numpy import pi

import pygsti
from pygsti.models import qutrit
from pygsti.algorithms.fiducialselection import find_fiducials
from pygsti.algorithms.germselection import find_germs
from pygsti.protocols import ProtocolData, StandardGST, StandardGSTDesign

## The target model

`create_qutrit_model` builds a target model for this situation: four operations on a single line label `T0`, three of them two-qubit unitaries projected onto the symmetric subspace. `Gx` is $X(\theta) \otimes X(\theta)$ and `Gy` is $Y(\theta) \otimes Y(\theta)$, one single-ion rotation applied identically to both ions at once. Applying the *same* rotation to both is what keeps the state inside the symmetric subspace, since any $U \otimes U$ commutes with the swap. `Gm` is the Mølmer-Sørensen unitary $\exp(-i \theta\, A \otimes A / 2)$ with $A = \cos\phi\, \sigma_x + \sin\phi\, \sigma_y$, which at `ms_local=0` reduces to $\exp(-i \theta\, \sigma_x \otimes \sigma_x / 2)$. `Gi` is the identity.

State preparation is $|00\rangle$, and the three POVM effects project onto $|00\rangle$, the symmetric combination of $|01\rangle$ and $|10\rangle$, and $|11\rangle$. Level $i$ is therefore $i$ bright ions, as the effect labels below show.

`basis="qt"` selects pyGSTi's qutrit basis, whose nine elements are two-qubit Pauli products projected onto that same symmetric subspace and orthonormalized under the trace inner product. The operator basis is thus built the way the gates are, and its labels are still written in two-qubit Pauli terms.

In [2]:
target_model = qutrit.create_qutrit_model(error_scale=0, x_angle=pi/2, y_angle=pi/2,
                                          ms_global=pi/2, ms_local=0, basis="qt")

print("state space:", target_model.state_space)
print("operations: ", list(target_model.operations.keys()))
print("effects:    ", list(target_model.povms['Mdefault'].keys()))
print("basis:      ", target_model.basis.labels)

state space: T0(9)
operations:  [Label(('Gi', 'T0')), Label(('Gx', 'T0')), Label(('Gy', 'T0')), Label(('Gm', 'T0'))]
effects:     ['0bright', '1bright', '2bright']
basis:       ['II', 'X+Y', 'X-Y', 'YZ', 'IX', 'IY', 'IZ', 'XY', 'XZ']


## Fiducials and germs

None of pyGSTi's current modelpacks describes a qutrit (the one that does, `stdQT_XYIMS`, sits in the deprecated `legacy` package), so this page searches for fiducials and germs rather than importing them. Both searches finish well under a second here. The candidate lists are all circuits up to a fixed length over the gate alphabet, so they grow exponentially in that length.

In [3]:
fiducialPrep, fiducialMeasure = find_fiducials(target_model, candidate_fid_counts={4: 'all upto'},
                                               algorithm='greedy')
germs = find_germs(target_model, randomize=False, candidate_germ_counts={4: 'all upto'},
                   mode='compactEVD', float_type=np.double)

Initial Length Available Fiducial List: 121
Length Available Fiducial List Dropped Identities and Duplicates: 50
Using greedy algorithm.
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.
Acceptable candidate solution found.
Score: major=-9 minor=17.999999999999996, N: 9
Exiting greedy search.
Preparation fiducials:
['{}@(T0)', 'Gm:T0Gm:T0@(T0)', 'Gx:T0@(T0)', 'Gy:T0@(T0)', 'Gx:T0Gx:T0Gx:T0@(T0)', 'Gy:T0Gy:T0Gy:T0@(T0)', 'Gm:T0@(T0)', 'Gm:T0Gx:T0@(T0)', 'Gm:T0Gy:T0@(T0)']
Score: 17.999999999999996
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.
Acceptable candidate solution found.
Score: major=-9 minor=7.971794871794874, N: 9
Exiting greedy search.
Measurement fiducials:
['{}@(T0)', 'Gx:T0@(T0)', 'Gm:T0@(T0)', 'Gm:T0Gx:T0@(T0)', 'Gy:T0@(T0)', 'Gy:T0Gm:T0@(T0)']
Score: 7.971794871794874
Initial Length Available Ger

In [4]:
print("%d prep fiducials" % len(fiducialPrep))
print("%d meas fiducials" % len(fiducialMeasure))
print("%d germs" % len(germs))

9 prep fiducials
6 meas fiducials
9 germs


## Circuits and data

`StandardGSTDesign` assembles the germ-power circuits sandwiched between fiducials. The dimension travels with the processor specification, which `create_processor_spec` reads off the model as one qudit of dimension three named `T0`.

In [5]:
maxLengths = [1, 2, 4]
design = StandardGSTDesign(target_model.create_processor_spec(), fiducialPrep, fiducialMeasure,
                           germs, maxLengths)
print("%d circuits" % len(design.all_circuits_needing_data))

700 circuits


Those circuits are what an experiment would have to run. Written out as an empty dataset they become a template: one row per circuit, three count columns to fill in.

In [6]:
pygsti.io.write_empty_dataset("../../../../example_files/dataTemplate_qutrit_maxL=4.txt",
                              design.all_circuits_needing_data,
                              "## Columns = 0bright count, 1bright count, 2bright count")

For a real experiment that template is the stopping point: take the data, fill the columns, and read the file back with `pygsti.io.load_dataset`. The rest of this page runs on simulated counts instead, drawn from a depolarized copy of the target model.

In [7]:
mdl_datagen = target_model.depolarize(op_noise=0.05, spam_noise=0.01)
DS = pygsti.data.simulate_data(mdl_datagen, design.all_circuits_needing_data,
                               num_samples=1000, sample_error='multinomial', seed=2018)
data = ProtocolData(design, DS)

## Running GST

`StandardGST` in `CPTPLND` mode fits a Lindblad-parameterized CPTP model. Nothing in the call is qutrit-specific; the three levels arrive with the model and the design.

The `optimizer` argument caps the Levenberg-Marquardt iterations at 50 to keep this page quick. Every stage then stops on the cap rather than on its own convergence test, and says so in the output below. Drop the cap when the answer matters.

In [8]:
result = StandardGST(modes=('CPTPLND',), target_model=target_model,
                     optimizer={'maxiter': 50}, verbosity=4).run(data)

-- Std Practice:  Iter 1 of 1  (CPTPLND) --: 
    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
  --- Iterative GST: Iter 1 of 3  186 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 4.22197e+06, mu=1, |x|=0, |J|=1854.09
      --- Outer Iter 1: norm_f = 393226, mu=82.1239, |x|=0.0810003, |J|=21356.8
      --- Outer Iter 2: norm_f = 22497.2, mu=27.3746, |x|=1.00193, |J|=1408.35
      --- Outer Iter 3: norm_f = 11825.4, mu=27.3746, |x|=1.3564, |J|=1102.51
      --- Outer Iter 4: norm_f = 3842.46, mu=25.1885, |x|=1.0641, |J|=1355.42
      --- Outer Iter 5: norm_f = 2882.2, mu=27.7091, |x|=1.02657, |J|=1383.63
      --- Outer Iter 6: norm_f = 2043.39, mu=29.0922, |x|=1.0485, |J|=1417.62
      --- Outer Iter 7: norm_f = 1070.31, mu=58.1707, |x|=0.973739, |J|=1494.1
      --- Outer Iter 8: norm_f = 729.584, mu=58.771

      --- Outer Iter 0: norm_f = 725.865, mu=1, |x|=0.851267, |J|=2251.14
      --- Outer Iter 1: norm_f = 645.779, mu=270.117, |x|=0.860276, |J|=2209.01
      --- Outer Iter 2: norm_f = 497.135, mu=133.276, |x|=0.83018, |J|=2244.72
      --- Outer Iter 3: norm_f = 476.854, mu=220.468, |x|=0.824717, |J|=2251.99
      --- Outer Iter 4: norm_f = 469.827, mu=216.243, |x|=0.823625, |J|=2252.98
      --- Outer Iter 5: norm_f = 464.791, mu=199.415, |x|=0.822687, |J|=2255.65
      --- Outer Iter 6: norm_f = 461.971, mu=169.177, |x|=0.822903, |J|=2257.76
      --- Outer Iter 7: norm_f = 460.606, mu=148.544, |x|=0.823824, |J|=2259.01
      --- Outer Iter 8: norm_f = 459.782, mu=130.115, |x|=0.825069, |J|=2259.89
      --- Outer Iter 9: norm_f = 459.23, mu=121.461, |x|=0.826819, |J|=2260.22
      --- Outer Iter 10: norm_f = 458.785, mu=118.035, |x|=0.828707, |J|=2260.3
      --- Outer Iter 11: norm_f = 458.406, mu=117.761, |x|=0.830924, |J|=2260.14
      --- Outer Iter 12: norm_f = 458.118, mu=1

      --- Outer Iter 0: norm_f = 1696.77, mu=1, |x|=0.847061, |J|=3605.61
      --- Outer Iter 1: norm_f = 1531.63, mu=698.276, |x|=0.865966, |J|=3526.71
      --- Outer Iter 2: norm_f = 1304.76, mu=323.396, |x|=0.841484, |J|=3589.46
      --- Outer Iter 3: norm_f = 1302.54, mu=560.824, |x|=0.841387, |J|=3581.26
      --- Outer Iter 4: norm_f = 1253.22, mu=186.941, |x|=0.830771, |J|=3607.59
      --- Outer Iter 5: norm_f = 1248.66, mu=306.08, |x|=0.829564, |J|=3606.23
      --- Outer Iter 6: norm_f = 1245.57, mu=204.627, |x|=0.827727, |J|=3607.22
      --- Outer Iter 7: norm_f = 1244.09, mu=193.892, |x|=0.826587, |J|=3606.57
      --- Outer Iter 8: norm_f = 1242.74, mu=147.927, |x|=0.82479, |J|=3608.07
      --- Outer Iter 9: norm_f = 1242, mu=140.069, |x|=0.823283, |J|=3608.45
      --- Outer Iter 10: norm_f = 1241.36, mu=106.755, |x|=0.821594, |J|=3609.93
      --- Outer Iter 11: norm_f = 1241.06, mu=104.315, |x|=0.820443, |J|=3610.5
      --- Outer Iter 12: norm_f = 1240.78, mu=87.6

      --- Outer Iter 0: norm_f = 621.076, mu=1, |x|=0.816135, |J|=2542
      --- Outer Iter 1: norm_f = 620.502, mu=118.335, |x|=0.815264, |J|=2544.79
      --- Outer Iter 2: norm_f = 620.437, mu=39.4451, |x|=0.815344, |J|=2544.81
      --- Outer Iter 3: norm_f = 620.418, mu=13.1484, |x|=0.815074, |J|=2544.76
      --- Outer Iter 4: norm_f = 620.416, mu=16.3088, |x|=0.814329, |J|=2544.45
      --- Outer Iter 5: norm_f = 620.403, mu=9.90382, |x|=0.813663, |J|=2544.61
      --- Outer Iter 6: norm_f = 620.398, mu=15.4815, |x|=0.813224, |J|=2544.59
      --- Outer Iter 7: norm_f = 620.395, mu=15.5265, |x|=0.812793, |J|=2544.39
      --- Outer Iter 8: norm_f = 620.391, mu=30.4914, |x|=0.812524, |J|=2544.44
      --- Outer Iter 9: norm_f = 620.389, mu=27.4463, |x|=0.812294, |J|=2544.39
      --- Outer Iter 10: norm_f = 620.387, mu=20.272, |x|=0.812065, |J|=2544.32
      --- Outer Iter 11: norm_f = 620.385, mu=17.504, |x|=0.811791, |J|=2544.19
      --- Outer Iter 12: norm_f = 620.383, mu=17.

## The report

The standard report needs nothing qutrit-specific either.

In [9]:
ws = pygsti.report.construct_standard_report(
    result, "Example Qutrit Report", verbosity=3
).write_html('../../../../example_files/sampleQutritReport', auto_open=False, verbosity=3)

Running idle tomography
Computing switchable properties


<repo>/pygsti/report/workspacetables.py:849: UserWarning: Failed gauge-robust decomposition of Gx:T0 op:
Cannot construct a real log: unpaired negative real eigenvalues: [np.complex128(-0.004972089994109194-1.9579202995006442e-16j)]
  _warnings.warn("Failed gauge-robust decomposition of %s op:\n%s" % (gl, str(e)))


Served with these docs: <a href="../../../../reports/sampleQutritReport/main.html">sampleQutritReport</a>.